# Задачи с сайта leetcode.com


In [1]:
import sqlite3

import pandas as pd

con = sqlite3.connect("sqlite.db")
cur = con.cursor()

In [2]:
def select(sql):
    return pd.read_sql(sql, con)


def crud(sql):
    cur.executescript(sql)
    con.commit()


def execute_formatted_sqlite(sql_schema):
    sql = ";\n".join(sql_schema.split("\n"))
    crud(sql)


### 175. Combine Two Tables (Easy)

Write a solution to report the first name, last name, city, and state of each person in the Person table. If the address of a personId is not present in the Address table, report null instead.

Return the result table in any order.


In [3]:
# SQLite3 не знает что такое TRUNCATE TABLE, поэтому нужно из запроса удалить эти команды.
# Добавить PRIMARY KEY к полям Id и OR REPLACE или IGNORE к INSERT, чтобы не дублировались строки
# при перезапуске. Или просто удалять таблици, если уже создана (DROP TABLE IF EXISTS table).

sql_schema = """
Drop table If Exists Person
Create table If Not Exists Person (personId int, firstName varchar(255), lastName varchar(255))
insert into Person (personId, lastName, firstName) values ('1', 'Wang', 'Allen')
insert into Person (personId, lastName, firstName) values ('2', 'Alice', 'Bob')
Drop table If Exists Address
Create table If Not Exists Address (addressId int, personId int, city varchar(255), state varchar(255))
insert into Address (addressId, personId, city, state) values ('1', '2', 'New York City', 'New York')
insert into Address (addressId, personId, city, state) values ('2', '3', 'Leetcode', 'California')
"""

# А так же после кажной строки поставить ';'
sql = ";\n".join(sql_schema.split("\n"))
print(sql)

# Так как несколько команд, нужен executescript(), а не execute()
cur.executescript(sql)

# В SQLite3 в Python нет автокомита (в CLI есть), поэтому:
con.commit()

;
Drop table If Exists Person;
Create table If Not Exists Person (personId int, firstName varchar(255), lastName varchar(255));
insert into Person (personId, lastName, firstName) values ('1', 'Wang', 'Allen');
insert into Person (personId, lastName, firstName) values ('2', 'Alice', 'Bob');
Drop table If Exists Address;
Create table If Not Exists Address (addressId int, personId int, city varchar(255), state varchar(255));
insert into Address (addressId, personId, city, state) values ('1', '2', 'New York City', 'New York');
insert into Address (addressId, personId, city, state) values ('2', '3', 'Leetcode', 'California');



In [4]:
sql = """--sql
SELECT
    p.firstName,
    p.lastName,
    a.city,
    a.state
FROM
    Person p
    LEFT JOIN Address a ON p.personId = a.personId;
"""
select(sql)

,firstName,lastName,city,state
0,Allen,Wang,None,None
1,Bob,Alice,New York City,New York


### 176. Second Highest Salary (Med.)

Write a solution to find the second highest distinct salary from the Employee table. If there is no second highest salary, return null (return None in Pandas).


In [5]:
sql_schema = """
Drop table If Exists Employee
Create table If Not Exists Employee (id int, salary int)
insert into Employee (id, salary) values ('1', '100')
insert into Employee (id, salary) values ('2', '200')
insert into Employee (id, salary) values ('3', '300')
"""
execute_formatted_sqlite(sql_schema)

In [6]:
# Если в таблице всего одна зарплата, OFFSET 1 не находит строк — запрос не возвращает вообще ничего.
# Вложив запрос в SELECT (...) — он вернёт строку, даже если внутри ничего не найдено.

sql = """--sql
SELECT
    (
        SELECT DISTINCT
            e.salary
        FROM
            Employee e
        ORDER BY
            e.salary DESC
        LIMIT
            1 OFFSET 1
    ) AS SecondHighestSalary;
"""
select(sql)

,SecondHighestSalary
0,200


### 177. Nth Highest Salary (Med.)

Write a solution to find the nth highest distinct salary from the Employee table. If there are less than n distinct salaries, return null.


In [7]:
sql_schema = """
Drop table If Exists Employee
Create table If Not Exists Employee (Id int, Salary int)
insert into Employee (id, salary) values ('1', '100')
insert into Employee (id, salary) values ('2', '200')
insert into Employee (id, salary) values ('3', '300')
"""
execute_formatted_sqlite(sql_schema)

In [8]:
# Для PostgreSQL
# CREATE
# OR REPLACE FUNCTION NthHighestSalary (n INT) RETURNS INT AS $$ -- начало тела функции
# SELECT
#     CASE
#         WHEN n < 1 THEN NULL
#         ELSE (
#             SELECT
#                 Salary
#             FROM
#                 (
#                     SELECT DISTINCT
#                         e.Salary
#                     FROM
#                         Employee e
#                     ORDER BY
#                         e.Salary DESC
#                     LIMIT
#                         1 OFFSET n -1
#                 ) AS result
#         )
#     END;

# $$ LANGUAGE SQL; -- конец тела функции с указанием используемого диалекта (чистый SQL)

# SELECT
#     NthHighestSalary (2);


# Имитация функции getNthHighestSalary для SQLite
def getNthHighestSalary(n):
    if n >= 1:
        sql = f"""--sql
        SELECT
            (
                SELECT DISTINCT
                    e.Salary
                FROM
                    Employee e
                ORDER BY
                    e.Salary desc
                LIMIT
                    1
                OFFSET
                    {n - 1}
            ) AS "getNthHighestSalary({n})";
        """
    else:
        sql = f"""--sql
        SELECT
            NULL AS "getNthHighestSalary({n})";
        """
    return select(sql)


getNthHighestSalary(-1)

,getNthHighestSalary(-1)
0,None


### 178. Rank Scores (Med.)

Write a solution to find the rank of the scores. The ranking should be calculated according to the following rules:

The scores should be ranked from the highest to the lowest.
If there is a tie between two scores, both should have the same ranking.
After a tie, the next ranking number should be the next consecutive integer value. In other words, there should be no holes between ranks.
Return the result table ordered by score in descending order.


In [9]:
sql_schema = """
Drop table If Exists Scores
Create table If Not Exists Scores (id int, score DECIMAL(3,2))
insert into Scores (id, score) values ('1', '3.5')
insert into Scores (id, score) values ('2', '3.65')
insert into Scores (id, score) values ('3', '4.0')
insert into Scores (id, score) values ('4', '3.85')
insert into Scores (id, score) values ('5', '4.0')
insert into Scores (id, score) values ('6', '3.65')
"""
execute_formatted_sqlite(sql_schema)

In [10]:
sql = """--sql
SELECT
    s.score,
    DENSE_RANK() OVER (
        ORDER BY
            s.score DESC
    ) AS RANK
FROM
    Scores s;
"""
select(sql)

,score,RANK
0,4.00,1
1,4.00,1
2,3.85,2
3,3.65,3
4,3.65,3
5,3.50,4


### 180. Consecutive Numbers (Med.)

Find all numbers that appear at least three times consecutively.


In [11]:
sql_schema = """
Drop table If Exists Logs
Create table If Not Exists Logs (id int, num int)
insert into Logs (id, num) values ('1', '1')
insert into Logs (id, num) values ('2', '1')
insert into Logs (id, num) values ('3', '1')
insert into Logs (id, num) values ('4', '2')
insert into Logs (id, num) values ('5', '1')
insert into Logs (id, num) values ('6', '2')
insert into Logs (id, num) values ('7', '2')
"""
execute_formatted_sqlite(sql_schema)

In [12]:
sql = """--sql
SELECT DISTINCT
    t.num AS ConsecutiveNums
FROM
    (
        SELECT
            l.num,
            LAG(l.num, 1) OVER (
                ORDER BY
                    id
            ) AS prev_num,
            LAG(l.num, 2) OVER (
                ORDER BY
                    id
            ) AS prev_num_2
        FROM
            Logs l
    ) t
WHERE
    t.num = t.prev_num
    AND t.num = t.prev_num_2;
"""
select(sql)

,ConsecutiveNums
0,1


### 181. Employees Earning More Than Their Managers (Easy)

Write a solution to find the employees who earn more than their managers.


In [13]:
sql_schema = """
Drop table If Exists Employee
Create table If Not Exists Employee (id int, name varchar(255), salary int, managerId int)
insert into Employee (id, name, salary, managerId) values ('1', 'Joe', '70000', '3')
insert into Employee (id, name, salary, managerId) values ('2', 'Henry', '80000', '4')
insert into Employee (id, name, salary, managerId) values ('3', 'Sam', '60000', NULL)
insert into Employee (id, name, salary, managerId) values ('4', 'Max', '90000', NULL)
"""
execute_formatted_sqlite(sql_schema)

In [14]:
sql = """--sql
SELECT
    e.name AS Employee
FROM
    Employee e
WHERE
    e.salary > (
        SELECT
            m.salary
        FROM
            Employee m
        WHERE
            m.id = e.managerId
    );
"""
select(sql)

,Employee
0,Joe


### 182. Duplicate Emails (Easy)

Write a solution to report all the duplicate emails. Note that it's guaranteed that the email field is not NULL.


In [15]:
sql_schema = """
Drop table If Exists Person
Create table If Not Exists Person (id int, email varchar(255))
insert into Person (id, email) values ('1', 'a@b.com')
insert into Person (id, email) values ('2', 'c@d.com')
insert into Person (id, email) values ('3', 'a@b.com')
"""
execute_formatted_sqlite(sql_schema)

In [16]:
sql = """--sql
SELECT
    e.email AS Email
FROM
    (
        SELECT
            p.email,
            COUNT(p.email) dupl
        FROM
            Person p
        GROUP BY
            p.email
    ) e
WHERE
    e.dupl > 1;
"""
select(sql)

,Email
0,a@b.com


### 183. Customers Who Never Order (Easy)

Write a solution to find all customers who never order anything.


In [17]:
sql_schema = """
Drop table If Exists Customers
Create table If Not Exists Customers (id int, name varchar(255))
insert into Customers (id, name) values ('1', 'Joe')
insert into Customers (id, name) values ('2', 'Henry')
insert into Customers (id, name) values ('3', 'Sam')
insert into Customers (id, name) values ('4', 'Max')
Drop table If Exists Orders
Create table If Not Exists Orders (id int, customerId int)
insert into Orders (id, customerId) values ('1', '3')
insert into Orders (id, customerId) values ('2', '1')
"""
execute_formatted_sqlite(sql_schema)

In [18]:
sql = """--sql
SELECT
    c.name AS Customers
FROM
    Customers c
WHERE
    c.id NOT IN (
        SELECT
            o.customerId
        FROM
            Orders o
    );
"""
select(sql)

,Customers
0,Henry
1,Max


### 184. Department Highest Salary (Med.)

Write a solution to find employees who have the highest salary in each of the departments.


In [19]:
sql_schema = """
Drop table If Exists Employee
Create table If Not Exists Employee (id int, name varchar(255), salary int, departmentId int)
insert into Employee (id, name, salary, departmentId) values ('1', 'Joe', '70000', '1')
insert into Employee (id, name, salary, departmentId) values ('2', 'Jim', '90000', '1')
insert into Employee (id, name, salary, departmentId) values ('3', 'Henry', '80000', '2')
insert into Employee (id, name, salary, departmentId) values ('4', 'Sam', '60000', '2')
insert into Employee (id, name, salary, departmentId) values ('5', 'Max', '90000', '1')
Drop table If Exists Department
Create table If Not Exists Department (id int, name varchar(255))
insert into Department (id, name) values ('1', 'IT')
insert into Department (id, name) values ('2', 'Sales')
"""
execute_formatted_sqlite(sql_schema)

In [20]:
sql = """--sql
SELECT
    d.name AS Department,
    e.name AS Employee,
    e.salary AS Salary
FROM
    (
        SELECT
            e.*,
            DENSE_RANK() OVER (
                PARTITION BY
                    e.departmentId
                ORDER BY
                    e.salary DESC
            ) AS rank
        FROM
            Employee e
    ) e
    LEFT JOIN Department d ON e.departmentId = d.id
WHERE
    e.rank = 1
ORDER BY
    d.name,
    e.name;
"""
select(sql)

,Department,Employee,Salary
0,IT,Jim,90000
1,IT,Max,90000
2,Sales,Henry,80000


### 185. Department Top Three Salaries (Hard)

A company's executives are interested in seeing who earns the most money in each of the company's departments. A high earner in a department is an employee who has a salary in the top three unique salaries for that department.

Write a solution to find the employees who are high earners in each of the departments.


In [21]:
sql_schema = """
Drop table If Exists Employee
Create table If Not Exists Employee (id int, name varchar(255), salary int, departmentId int)
insert into Employee (id, name, salary, departmentId) values ('1', 'Joe', '85000', '1')
insert into Employee (id, name, salary, departmentId) values ('2', 'Henry', '80000', '2')
insert into Employee (id, name, salary, departmentId) values ('3', 'Sam', '60000', '2')
insert into Employee (id, name, salary, departmentId) values ('4', 'Max', '90000', '1')
insert into Employee (id, name, salary, departmentId) values ('5', 'Janet', '69000', '1')
insert into Employee (id, name, salary, departmentId) values ('6', 'Randy', '85000', '1')
insert into Employee (id, name, salary, departmentId) values ('7', 'Will', '70000', '1')
Drop table If Exists Department
Create table If Not Exists Department (id int, name varchar(255))
insert into Department (id, name) values ('1', 'IT')
insert into Department (id, name) values ('2', 'Sales')
"""
execute_formatted_sqlite(sql_schema)

In [22]:
sql = """--sql
SELECT
    d.name AS Department,
    e.name AS Employee,
    e.salary AS Salary
FROM
    (
        SELECT
            e.*,
            DENSE_RANK() OVER (
                PARTITION BY
                    e.departmentId
                ORDER BY
                    e.salary DESC
            ) AS rank
        FROM
            Employee e
    ) e
    LEFT JOIN Department d ON e.departmentId = d.id
WHERE
    e.rank <= 3
ORDER BY
    d.name,
    e.salary DESC;
"""
select(sql)

,Department,Employee,Salary
0,IT,Max,90000
1,IT,Joe,85000
2,IT,Randy,85000
3,IT,Will,70000
4,Sales,Henry,80000
5,Sales,Sam,60000


### 196. Delete Duplicate Emails (Easy)

Write a solution to delete all duplicate emails, keeping only one unique email with the smallest id.

For SQL users, please note that you are supposed to write a DELETE statement and not a SELECT one.

For Pandas users, please note that you are supposed to modify Person in place.

After running your script, the answer shown is the Person table. The driver will first compile and run your piece of code and then show the Person table. The final order of the Person does not matter.


In [23]:
sql_schema = """
Drop table If Exists Person
Create table If Not Exists Person (Id int, Email varchar(255))
insert into Person (id, email) values ('1', 'john@example.com')
insert into Person (id, email) values ('2', 'bob@example.com')
insert into Person (id, email) values ('3', 'john@example.com')
"""
execute_formatted_sqlite(sql_schema)

In [24]:
sql = """--sql
DELETE
FROM
    Person
WHERE
    id NOT IN (
        SELECT
            MIN(p.id)
        FROM
            Person p
        GROUP BY
            p.email
    );
"""
crud(sql)

### 197. Rising Temperature (Easy)

Write a solution to find all dates' id with higher temperatures compared to its previous dates (yesterday).
